# Tuần 3 — Huấn luyện `ViT-LoRA + ViTDetBoxHead` & Đối sánh 3 Kênh `[FLAIR, T1ce, T2]` vs 1 Kênh `T1ce` trên Dataset 4 (BraTS2020)

**Thành viên phụ trách:** Lâm Gia Bảo *(bản đã sửa lỗi `num_classes=4 -> binary 2 lớp`, loại bỏ toàn bộ số liệu hardcode & bổ sung auto-resume từng epoch)*

### Hai nhiệm vụ thực nghiệm trên Dataset 4 (BraTS2020 — 42.034 lát cắt đa xung):
1. **Nhiệm vụ 1:** Huấn luyện **`ViT-LoRA (r=8, α=16)` + `ViTDetBoxHead` (`GIoU + L1 Loss`)** trên tập lát cắt 3 kênh **`[FLAIR, T1ce, T2]`** (`29.772 train / 6.087 val / 6.175 test`, 0% rò rỉ bệnh nhân).
2. **Nhiệm vụ 2:** Chạy thực nghiệm đối sánh công bằng (cùng seed, cùng chia tập, cùng kiến trúc & số epoch) giữa **3 kênh `[FLAIR, T1ce, T2]`** so với **1 kênh `T1ce` đơn lẻ** trên cả khả năng **phát hiện khối u (`Accuracy`, `Recall`, `Specificity`, `F1`, `ROC-AUC`)** và **đóng khung khối u (`Mean Box IoU`, `IoU@0.50`, `IoU@0.75`)**.

### Cách chạy trên Google Colab:
1. Chọn **`Runtime -> Change runtime type -> T4 GPU`** (hoặc `L4 / A100 GPU`).
2. Đặt đường dẫn file `BraTS2020_png.zip` trong Google Drive ở **Cell 1**.
3. Chạy lần lượt từ trên xuống dưới (`Ctrl + F9`). Nếu Colab ngắt kết nối giữa chừng, chạy lại từ đầu sẽ **tự động resume từ epoch gần nhất** đã lưu trên Drive.

---
## Cell 1: Mount Google Drive & Cấu hình Đường dẫn Dữ liệu / Lưu trữ

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Đang chạy trên Google Colab: {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ZIP = "/content/drive/MyDrive/BrainTumorDetect/dataset/BraTS2020_png.zip"
    DRIVE_OUT = "/content/drive/MyDrive/BrainTumorDetect/week3_brats2020_bao"
    DRIVE_TEAM = "/content/drive/MyDrive/BrainTumorDetect/team_results/LamGiaBao_Week3"
    os.makedirs(DRIVE_OUT, exist_ok=True)
    os.makedirs(DRIVE_TEAM, exist_ok=True)
    if os.path.exists(DRIVE_ZIP):
        print(f"✅ Tìm thấy file zip: {DRIVE_ZIP} ({os.path.getsize(DRIVE_ZIP) / 1e9:.2f} GB)")
    else:
        print(f"⚠️ Không thấy file zip tại {DRIVE_ZIP}. Đang tìm tự động trong Drive...")
        res = subprocess.run(["find", "/content/drive/MyDrive", "-name", "BraTS2020_png.zip", "-maxdepth", "4"],
                             capture_output=True, text=True)
        if res.stdout.strip():
            DRIVE_ZIP = res.stdout.strip().splitlines()[0]
            print(f"🔍 Tự động chọn: {DRIVE_ZIP}")
        else:
            raise FileNotFoundError("Không tìm thấy BraTS2020_png.zip trên Google Drive! Hãy cập nhật biến DRIVE_ZIP.")
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
    DRIVE_ZIP = ""
    DRIVE_OUT = str(PROJECT_ROOT / "runs" / "week3_brats2020_bao")
    DRIVE_TEAM = str(PROJECT_ROOT / "reports" / "tuan3" / "vit_lora_bao")
    os.makedirs(DRIVE_OUT, exist_ok=True)
    os.makedirs(DRIVE_TEAM, exist_ok=True)
    print(f"Chạy cục bộ tại {PROJECT_ROOT}, lưu kết quả ở {DRIVE_OUT}")

---
## Cell 2: Cài đặt Thư viện (Gỡ xung đột `torchao`), Clone Repo & Giải nén BraTS2020

In [ ]:
import time
import zipfile

if IN_COLAB:
    # Gỡ torchao 0.10.0 cài sẵn trên Colab gây lỗi ImportError với peft mới
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "peft>=0.10.0", "transformers>=4.40.0", "timm>=0.9.16",
        "albumentations>=1.4.0", "scipy", "scikit-learn>=1.4.0",
        "opencv-python-headless", "tqdm"
    ])
    PROJECT_ROOT = Path("/content/BrainTumorDetect")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.check_call(["git", "clone", "-b", "dev", "https://github.com/MikanQuyt/BrainTumorDetect.git", str(PROJECT_ROOT)])
    os.chdir(PROJECT_ROOT)
else:
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DEST = PROJECT_ROOT / "dataset" / "brats2020_png"
if IN_COLAB and (not (DEST / "images").is_dir() or len(os.listdir(DEST / "images")) < 1000):
    print(f"📦 Đang giải nén {DRIVE_ZIP} -> {DEST} ...")
    t0 = time.time()
    (PROJECT_ROOT / "dataset").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zf:
        first = zf.namelist()[0]
    if first.startswith("BraTS2020_png/") or first.startswith("brats2020_png/"):
        subprocess.check_call(["unzip", "-q", "-o", DRIVE_ZIP, "-d", str(PROJECT_ROOT / "dataset")])
        alt = PROJECT_ROOT / "dataset" / "BraTS2020_png"
        if alt.is_dir() and not DEST.is_dir():
            alt.rename(DEST)
    else:
        DEST.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(["unzip", "-q", "-o", DRIVE_ZIP, "-d", str(DEST)])
    print(f"✅ Giải nén xong sau {time.time() - t0:.1f}s")

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")

---
## Cell 3: Tạo & Kiểm định `dataset/brats2020_annotations.json` (Chống Rò rỉ Bệnh nhân)
- Khớp nối trực tiếp với `metadata.csv` của thư mục `dataset/brats2020_png`.
- Kiểm tra 100% ảnh tồn tại trên đĩa và kiểm định **0 bệnh nhân (`case_id`) giao nhau giữa `train`, `val`, `test`**.

In [ ]:
import json
import pandas as pd

ANN_FILE = PROJECT_ROOT / "dataset" / "brats2020_annotations.json"
META_CSV = DEST / "metadata.csv"

if META_CSV.exists():
    print(f"📄 Đồng bộ annotations từ {META_CSV} ...")
    df_meta = pd.read_csv(META_CSV)
    records = []
    for _, row in df_meta.iterrows():
        has_tumor = int(row["has_tumor"])
        gt_box = None
        if has_tumor and pd.notna(row.get("x_min")):
            gt_box = [int(row["x_min"]), int(row["y_min"]), int(row["x_max"]) + 1, int(row["y_max"]) + 1]
        png_p = str(row.get("png_path", f"images/{row['case_id']}_z{int(row['z']):03d}.png"))
        msk_p = str(row.get("mask_path", f"masks/{row['case_id']}_z{int(row['z']):03d}.png"))
        records.append({
            "dataset": "brats2020",
            "case_id": str(row["case_id"]),
            "slice": int(row["z"]),
            "img_path": f"dataset/brats2020_png/{png_p}",
            "mask_path": f"dataset/brats2020_png/{msk_p}",
            "has_mask": True,
            "has_tumor": has_tumor,
            "sub_class_label": 1 if has_tumor else 0,
            "gt_box": gt_box,
            "split": str(row["split"]),
            "fold": None,
        })
    ANN_FILE.write_text(json.dumps(records, indent=2, ensure_ascii=False), encoding="utf-8")
else:
    records = json.loads(ANN_FILE.read_text(encoding="utf-8-sig"))

# Kiểm định không thiếu file và không rò rỉ case_id
missing = [r["img_path"] for r in records if not (PROJECT_ROOT / r["img_path"]).exists()]
cases_by_split = {sp: {r["case_id"] for r in records if r["split"] == sp} for sp in ("train", "val", "test")}
leak_tv = cases_by_split["train"] & cases_by_split["val"]
leak_tt = cases_by_split["train"] & cases_by_split["test"]
leak_vt = cases_by_split["val"] & cases_by_split["test"]
assert len(missing) == 0, f"Thiếu {len(missing)} ảnh trên đĩa! Ví dụ: {missing[:3]}"
assert not (leak_tv or leak_tt or leak_vt), "Phát hiện rò rỉ case_id giữa các tập!"

df_stat = pd.DataFrame([{"split": r["split"], "label": "tumor" if r["has_tumor"] else "no_tumor", "case_id": r["case_id"]} for r in records])
print(f"✅ Tổng {len(records):,} lát cắt hợp lệ | 0 ảnh thiếu | 0 rò rỉ bệnh nhân")
display(pd.crosstab(df_stat["split"], df_stat["label"], margins=True))

---
## Cell 4: Cấu hình Thống nhất (`task='binary', num_classes=2`) & Lớp bọc `T1ceOnlyDataset`
**Khắc phục triệt để lỗi gốc:**
- BraTS2020 chỉ có 2 nhãn (`0: no_tumor`, `1: tumor / glioma`). Việc đặt **`task='binary', num_classes=2`** đảm bảo `compute_class_weights` tính đúng trọng số cân bằng (~`1.08` vs `0.92`) và hoàn toàn không bị sụp `Acc = F1 = Recall = 0` như bản cũ (`num_classes=4`).
- Lớp `T1ceOnlyDataset` trích xuất kênh **`T1ce` gốc (kênh index `1`, khử chuẩn hóa ImageNet của kênh `G` trước rồi mới nhân bản 3 kênh và chuẩn hóa lại theo ImageNet)** giúp thí nghiệm đối sánh `3-ch [FLAIR, T1ce, T2]` vs `1-ch T1ce` công bằng tuyệt đối.

In [ ]:
import csv
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset

from data.unified_dataset import IMAGENET_MEAN, IMAGENET_STD, UnifiedBrainMRIDataset
from models.vit_lora import ViTLoRA
from utils.losses import BoxRegressionLoss, build_classification_loss, compute_class_weights
from utils.metrics import BoxLocalizationMeter, ClassificationMeter, detection_metrics, optimize_detection_threshold

CFG = dict(
    model_name="google/vit-base-patch16-224-in21k",
    lora_r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query", "value"],
    use_box_head=True, sfp_dim=128, dropout=0.1,
    # QUAN TRỌNG: BraTS2020 chỉ có 2 nhãn {0: no_tumor, 1: tumor/glioma}
    task="binary", num_classes=2,
    # Cấu hình tối ưu cho Google Colab T4 (công bằng cho cả 3-ch và 1-ch T1ce)
    epochs=20, patience=6, batch_size=64, num_workers=2,
    lr_lora=3e-4, lr_head=1e-3, lr_box=1e-3, weight_decay=0.01,
    loss_type="focal", focal_gamma=2.0, label_smoothing=0.05,
    giou_weight=1.0, l1_weight=1.0, box_loss_weight=1.0,
    threshold=0.35,
    limit_train=0,   # Đặt 0 để train toàn bộ 29.772 lát cắt; đặt 6000 nếu muốn chạy thử nhanh
    limit_eval=0,    # Đặt 0 để eval toàn bộ 6.087 val / 6.175 test
    seed=42, amp=torch.cuda.is_available(),
    device="cuda" if torch.cuda.is_available() else "cpu",
    ann_files=["dataset/brats2020_annotations.json"],
    datasets=["brats2020"],
)
CLASS_NAMES = ["no_tumor", "tumor"]

def seed_everything(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def build_brats_ds(split: str, train: bool) -> UnifiedBrainMRIDataset:
    ds = UnifiedBrainMRIDataset(CFG["ann_files"], train=train, datasets=CFG["datasets"], split=split)
    ds.records = [r for r in ds.records if int(r["has_tumor"]) in (0, 1)]
    lim = CFG["limit_train"] if train else CFG["limit_eval"]
    if lim and len(ds.records) > lim:
        rng = np.random.RandomState(CFG["seed"])
        keep = sorted(rng.choice(len(ds.records), lim, replace=False))
        ds.records = [ds.records[i] for i in keep]
    return ds

class T1ceOnlyDataset(Dataset):
    """Trích xuất kênh T1ce gốc (channel 1), nhân bản 3 kênh rồi chuẩn hóa lại đúng chuẩn ImageNet."""
    def __init__(self, base: UnifiedBrainMRIDataset):
        self.base = base
        self.mean = torch.tensor(IMAGENET_MEAN, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(IMAGENET_STD, dtype=torch.float32).view(3, 1, 1)

    def __len__(self): return len(self.base)

    def __getitem__(self, idx):
        s = self.base[idx]
        # Khôi phục giá trị [0, 1] của kênh T1ce (index 1) rồi chuẩn hóa lại đều cho 3 kênh
        t1ce_01 = (s["image_224"][1:2] * IMAGENET_STD[1] + IMAGENET_MEAN[1]).clamp(0.0, 1.0)
        s["image_224"] = (t1ce_01.expand(3, -1, -1) - self.mean) / self.std
        s["image_1024"] = s["image_1024"][1:2].expand(3, -1, -1).contiguous()
        return s

def make_loaders(t1ce_only: bool = False):
    tr_ds = build_brats_ds("train", True)
    va_ds = build_brats_ds("val", False)
    ts_ds = build_brats_ds("test", False)
    tr_labels = [int(r["has_tumor"]) for r in tr_ds.records]
    if t1ce_only:
        tr_ds, va_ds, ts_ds = T1ceOnlyDataset(tr_ds), T1ceOnlyDataset(va_ds), T1ceOnlyDataset(ts_ds)
    g = torch.Generator().manual_seed(CFG["seed"])
    kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"],
              pin_memory=CFG["device"].startswith("cuda"), drop_last=False)
    return (
        DataLoader(tr_ds, shuffle=True, generator=g, **kw),
        DataLoader(va_ds, shuffle=False, **kw),
        DataLoader(ts_ds, shuffle=False, **kw),
        tr_labels,
    )

# Kiểm tra class_weights với 2 lớp (phải cân bằng ~1.0, tuyệt đối không có lớp rỗng)
_, _, _, _lbls = make_loaders(t1ce_only=False)
cw_preview = compute_class_weights(_lbls, num_classes=2)
print(f"✅ Kiểm tra Class Weights (2 lớp no_tumor/tumor): {cw_preview.tolist()} (trung bình = {cw_preview.mean():.4f})")

---
## Cell 5: Hàm Huấn luyện, Đánh giá & Auto-Resume Mỗi Epoch (Lưu trực tiếp lên Google Drive)

In [ ]:
def run_epoch(model, loader, cls_crit, box_crit, optimizer=None, scaler=None, desc=""):
    training = optimizer is not None
    model.train(training)
    cm = ClassificationMeter(CLASS_NAMES)
    lm = BoxLocalizationMeter()
    tl, cl, bl, n = 0.0, 0.0, 0.0, 0
    dev = CFG["device"]
    amp = CFG["amp"] and dev.startswith("cuda")
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for step, batch in enumerate(loader, 1):
            x = batch["image_224"].to(dev, non_blocking=True)
            y = batch["binary_label"].to(dev, non_blocking=True)
            gb = batch["gt_box_norm"].to(dev, non_blocking=True)
            hb = batch["has_box"].to(dev, non_blocking=True)

            with torch.autocast(device_type=dev.split(":")[0], dtype=torch.float16, enabled=amp):
                out = model(x, return_dict=True)
                logits = out["logits"]
                c_loss = cls_crit(logits.float(), y)
                br = box_crit(out["pred_boxes"], gb, has_box=hb)
                b_loss = br["loss"]
                loss = c_loss + CFG["box_loss_weight"] * b_loss

            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            bs = len(y)
            tl += float(loss.detach()) * bs
            cl += float(c_loss.detach()) * bs
            bl += float(b_loss.detach()) * bs
            n += bs

            probs = logits.detach().float().softmax(dim=1).cpu().numpy()
            cm.update(y.cpu().numpy(), probs)

            pb = out["pred_boxes"].detach().cpu().numpy()
            gtb = gb.detach().cpu().numpy()
            hbb = hb.detach().cpu().numpy()
            for i in range(bs):
                if hbb[i]:
                    lm.update(pred_box=pb[i], gt_box=gtb[i])

            if step % 150 == 0:
                print(f"  {desc} step {step}/{len(loader)} loss={tl/n:.4f} (cls={cl/n:.4f}, box={bl/n:.4f})", flush=True)

    m = cm.compute(with_detection=True)
    ya, pa = cm.arrays()
    m["detection_tau_tuned"] = detection_metrics(ya, pa, threshold=CFG["threshold"])
    m["loss"] = tl / max(n, 1)
    m["cls_loss"] = cl / max(n, 1)
    m["box_loss"] = bl / max(n, 1)
    if lm.ious:
        m["box_head_localization"] = lm.summary()
    return m, cm


def train_and_evaluate(exp_name: str, t1ce_only: bool, out_dir: Path):
    """Huấn luyện + tự động resume từng epoch + đánh giá tập Test thật cho 1 cấu hình."""
    seed_everything(CFG["seed"])
    out_dir.mkdir(parents=True, exist_ok=True)
    train_loader, val_loader, test_loader, tr_labels = make_loaders(t1ce_only=t1ce_only)

    class_weights = compute_class_weights(tr_labels, num_classes=CFG["num_classes"])
    cls_crit = build_classification_loss(CFG["loss_type"], class_weights, CFG["focal_gamma"], CFG["label_smoothing"]).to(CFG["device"])
    box_crit = BoxRegressionLoss(CFG["giou_weight"], CFG["l1_weight"]).to(CFG["device"])

    model = ViTLoRA(
        num_classes=CFG["num_classes"], lora_r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"],
        lora_dropout=CFG["lora_dropout"], target_modules=CFG["target_modules"],
        use_box_head=CFG["use_box_head"], sfp_dim=CFG["sfp_dim"], dropout=CFG["dropout"],
        model_name=CFG["model_name"],
    ).to(CFG["device"])
    counts = model.param_counts()
    print(f"\n{'='*72}")
    print(f"🚀 {exp_name} | Train={len(train_loader.dataset):,} Val={len(val_loader.dataset):,} Test={len(test_loader.dataset):,}")
    print(f"   Tham số khả huấn: {counts['trainable']:,}/{counts['total']:,} ({counts['trainable_pct']:.2f}%)")
    print(f"{'='*72}")

    optimizer = torch.optim.AdamW(
        model.param_groups(lr_lora=CFG["lr_lora"], lr_head=CFG["lr_head"], lr_box=CFG["lr_box"], weight_decay=CFG["weight_decay"])
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"], eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if (CFG["amp"] and CFG["device"].startswith("cuda")) else None

    resume_ckpt = out_dir / "resume_ckpt.pt"
    best_ckpt = out_dir / "best_lora.pt"
    start_ep, best_score, best_ep, bad, history = 1, -1.0, 0, 0, []

    if resume_ckpt.exists():
        ck = torch.load(resume_ckpt, map_location=CFG["device"], weights_only=False)
        model.load_state_dict(ck["model_state_dict"], strict=False)
        optimizer.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        if scaler is not None and ck.get("scaler_state_dict"):
            scaler.load_state_dict(ck["scaler_state_dict"])
        start_ep = int(ck["epoch"]) + 1
        best_score, best_ep, bad = float(ck["best_score"]), int(ck["best_epoch"]), int(ck["bad"])
        history = ck.get("history", [])
        print(f"🔄 Resume từ epoch {ck['epoch']} (best_ep={best_ep}, best_score={best_score:.4f}, bad={bad}/{CFG['patience']})")

    t0 = time.time()
    for epoch in range(start_ep, CFG["epochs"] + 1):
        et0 = time.time()
        tr, _ = run_epoch(model, train_loader, cls_crit, box_crit, optimizer=optimizer, scaler=scaler, desc=f"[{exp_name} E{epoch} tr]")
        va, _ = run_epoch(model, val_loader, cls_crit, box_crit, desc=f"[{exp_name} E{epoch} va]")
        scheduler.step()

        vbi = va.get("box_head_localization", {}).get("mean_box_iou", float("nan"))
        score = va["f1"] + (0.2 * vbi if not np.isnan(vbi) else 0.0)
        et = time.time() - et0
        row = dict(
            epoch=epoch, train_loss=tr["loss"], train_cls=tr["cls_loss"], train_box=tr["box_loss"],
            val_loss=va["loss"], val_cls=va["cls_loss"], val_box=va["box_loss"],
            val_acc=va["accuracy"], val_f1=va["f1"], val_recall=va["recall"], val_auc=va["auc"],
            val_box_iou=vbi, lr=optimizer.param_groups[0]["lr"], secs=round(et, 1),
        )
        history.append(row)

        mark = ""
        if score > best_score:
            best_score, best_ep, bad = score, epoch, 0
            model.save_lora_checkpoint(best_ckpt, extra={"epoch": epoch, "val_f1": va["f1"], "val_box_iou": vbi})
            mark = " ⭐ BEST"
        else:
            bad += 1

        print(
            f"E{epoch:02d}/{CFG['epochs']} | loss={tr['loss']:.4f}/{va['loss']:.4f} | "
            f"acc={va['accuracy']:.4f} rec={va['recall']:.4f} f1={va['f1']:.4f} auc={va['auc']:.4f} | "
            f"box_iou={vbi:.4f} | {et:.0f}s{mark}", flush=True
        )
        # Lưu resume checkpoint & history.csv sau MỖI epoch để chống mất tiến độ
        pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
        torch.save({
            "epoch": epoch, "best_score": best_score, "best_epoch": best_ep, "bad": bad,
            "model_state_dict": model.trainable_state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "history": history,
        }, resume_ckpt)

        if bad >= CFG["patience"]:
            print(f"🛑 Dừng sớm tại epoch {epoch} (best epoch = {best_ep})")
            break

    # Đánh giá trên tập TEST độc lập với trọng số tốt nhất
    ck_best = torch.load(best_ckpt, map_location=CFG["device"], weights_only=False)
    model.load_state_dict(ck_best["state_dict"], strict=False)
    test_metrics, test_cm = run_epoch(model, test_loader, cls_crit, box_crit, desc=f"[{exp_name} TEST]")
    ya, pa = test_cm.arrays()
    test_metrics["best_epoch"] = best_ep
    test_metrics["params"] = counts
    test_metrics["threshold_optimization"] = optimize_detection_threshold(ya, pa, target_recall=0.95)["best"]
    test_metrics["elapsed_min"] = round((time.time() - t0) / 60.0, 2)

    (out_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    np.savez_compressed(out_dir / "test_predictions.npz", y_true=ya, y_prob=pa)
    loc = test_metrics.get("box_head_localization", {})
    det = test_metrics["detection"]
    print(f"\n✅ TEST [{exp_name}] (best ep {best_ep}): Acc={test_metrics['accuracy']*100:.2f}% | "
          f"Rec={test_metrics['recall']*100:.2f}% | Spec={det['specificity']*100:.2f}% | "
          f"F1={test_metrics['f1']*100:.2f}% | AUC={test_metrics['auc']:.4f} | "
          f"BoxIoU={loc.get('mean_box_iou', 0)*100:.2f}% | IoU@50={loc.get('iou_at_50', 0)*100:.2f}%")
    return model, test_metrics, pd.DataFrame(history)

---
## Cell 6: Thí nghiệm 1 — Huấn luyện & Đánh giá 3 Kênh `[FLAIR, T1ce, T2]` trên BraTS2020

In [ ]:
OUT_3CH = Path(DRIVE_OUT) / "exp_3ch_flair_t1ce_t2"
model_3ch, test_3ch, hist_3ch = train_and_evaluate("3ch_FLAIR_T1ce_T2", t1ce_only=False, out_dir=OUT_3CH)

# Vẽ đường cong hội tụ 3 kênh
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(hist_3ch["epoch"], hist_3ch["train_loss"], marker="o", label="Train Loss")
axes[0].plot(hist_3ch["epoch"], hist_3ch["val_loss"], marker="s", label="Val Loss")
axes[0].set_title("Total Loss (3 kênh)", fontweight="bold"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(hist_3ch["epoch"], hist_3ch["val_acc"] * 100, marker="o", label="Val Acc (%)")
axes[1].plot(hist_3ch["epoch"], hist_3ch["val_f1"] * 100, marker="s", label="Val F1 (%)")
axes[1].plot(hist_3ch["epoch"], hist_3ch["val_recall"] * 100, marker="^", label="Val Recall (%)")
axes[1].set_title("Phân loại Validation (3 kênh)", fontweight="bold"); axes[1].legend(); axes[1].grid(True, alpha=0.4)

axes[2].plot(hist_3ch["epoch"], hist_3ch["val_box_iou"] * 100, marker="D", color="teal", label="Val Mean Box IoU (%)")
axes[2].set_title("Đóng khung Box IoU (3 kênh)", fontweight="bold"); axes[2].legend(); axes[2].grid(True, alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_3CH / "curves_3ch.png", dpi=180, bbox_inches="tight")
plt.show()

---
## Cell 7: Thí nghiệm 2 — Huấn luyện & Đánh giá 1 Kênh `T1ce` Đơn lẻ trên BraTS2020

In [ ]:
OUT_T1CE = Path(DRIVE_OUT) / "exp_1ch_t1ce_only"
model_t1ce, test_t1ce, hist_t1ce = train_and_evaluate("1ch_T1ce_only", t1ce_only=True, out_dir=OUT_T1CE)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(hist_t1ce["epoch"], hist_t1ce["train_loss"], marker="o", label="Train Loss")
axes[0].plot(hist_t1ce["epoch"], hist_t1ce["val_loss"], marker="s", label="Val Loss")
axes[0].set_title("Total Loss (1 kênh T1ce)", fontweight="bold"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(hist_t1ce["epoch"], hist_t1ce["val_acc"] * 100, marker="o", label="Val Acc (%)")
axes[1].plot(hist_t1ce["epoch"], hist_t1ce["val_f1"] * 100, marker="s", label="Val F1 (%)")
axes[1].plot(hist_t1ce["epoch"], hist_t1ce["val_recall"] * 100, marker="^", label="Val Recall (%)")
axes[1].set_title("Phân loại Validation (1 kênh T1ce)", fontweight="bold"); axes[1].legend(); axes[1].grid(True, alpha=0.4)

axes[2].plot(hist_t1ce["epoch"], hist_t1ce["val_box_iou"] * 100, marker="D", color="coral", label="Val Mean Box IoU (%)")
axes[2].set_title("Đóng khung Box IoU (1 kênh T1ce)", fontweight="bold"); axes[2].legend(); axes[2].grid(True, alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_T1CE / "curves_t1ce.png", dpi=180, bbox_inches="tight")
plt.show()

---
## Cell 8: Đối sánh Thực nghiệm 3 Kênh `[FLAIR, T1ce, T2]` vs 1 Kênh `T1ce` (100% Từ Tập Test Thật)
Tạo bảng đối sánh, biểu đồ cột, đường cong hội tụ so sánh và lưới ảnh định tính so sánh hộp dự đoán trên 6 lát cắt u não thuộc tập Test.

In [ ]:
import matplotlib.patches as patches
from utils.metrics import box_iou_score

def summarize_row(m: dict, name: str) -> dict:
    loc = m.get("box_head_localization", {})
    det = m.get("detection", {})
    return {
        "Config": name,
        "Best_Epoch": int(m.get("best_epoch", 0)),
        "Test_Loss": round(float(m["loss"]), 5),
        "Accuracy (%)": round(float(m["accuracy"]) * 100, 2),
        "Recall (%)": round(float(m["recall"]) * 100, 2),
        "Specificity (%)": round(float(det.get("specificity", 0.0)) * 100, 2),
        "F1 (%)": round(float(m["f1"]) * 100, 2),
        "ROC_AUC": round(float(m["auc"]), 4),
        "Mean_Box_IoU (%)": round(float(loc.get("mean_box_iou", 0.0)) * 100, 2),
        "IoU@0.50 (%)": round(float(loc.get("iou_at_50", 0.0)) * 100, 2),
        "IoU@0.75 (%)": round(float(loc.get("iou_at_75", 0.0)) * 100, 2),
    }

r3 = summarize_row(test_3ch, "3-ch [FLAIR, T1ce, T2]")
r1 = summarize_row(test_t1ce, "1-ch T1ce only")
comp_df = pd.DataFrame([r3, r1])
display(comp_df)

out_root = Path(DRIVE_OUT)
comp_df.to_csv(out_root / "comparison_3ch_vs_t1ce.csv", index=False)
(out_root / "comparison_3ch_vs_t1ce.json").write_text(
    json.dumps({"3ch": test_3ch, "1ch_t1ce": test_t1ce, "summary": [r3, r1]}, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

# 1. Biểu đồ cột đối sánh Test Set
metric_keys = ["Accuracy (%)", "Recall (%)", "Specificity (%)", "F1 (%)", "Mean_Box_IoU (%)", "IoU@0.50 (%)", "IoU@0.75 (%)"]
v3 = [r3[k] for k in metric_keys]
v1 = [r1[k] for k in metric_keys]
x = np.arange(len(metric_keys)); w = 0.36
fig, ax = plt.subplots(figsize=(12, 5.2))
b3 = ax.bar(x - w/2, v3, w, label="3 kênh [FLAIR, T1ce, T2]", color="#2563EB", zorder=3)
b1 = ax.bar(x + w/2, v1, w, label="1 kênh T1ce đơn lẻ", color="#F97316", zorder=3)
for bars in (b3, b1):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8, f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8.5, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([k.replace(" (%)", "") for k in metric_keys], fontweight="bold")
ax.set_ylim(0, 108); ax.set_ylabel("Tỷ lệ (%)"); ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.set_title("Đối sánh Thực nghiệm trên BraTS2020 Test Set: 3 Kênh [FLAIR, T1ce, T2] vs 1 Kênh T1ce", fontweight="bold", fontsize=12)
ax.legend(loc="lower right", fontsize=10)
plt.tight_layout()
fig.savefig(out_root / "comparison_3ch_vs_t1ce.png", dpi=200)
plt.show()

# 2. Đường cong hội tụ đối sánh theo Epoch
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].plot(hist_3ch["epoch"], hist_3ch["val_f1"] * 100, marker="o", color="#2563EB", lw=2.2, label="3-ch [FLAIR, T1ce, T2]")
axes[0].plot(hist_t1ce["epoch"], hist_t1ce["val_f1"] * 100, marker="s", color="#F97316", lw=2.2, ls="--", label="1-ch T1ce only")
axes[0].set_title("(a) Validation F1-Score theo Epoch", fontweight="bold"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("F1 (%)"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

axes[1].plot(hist_3ch["epoch"], hist_3ch["val_box_iou"] * 100, marker="o", color="#2563EB", lw=2.2, label="3-ch [FLAIR, T1ce, T2]")
axes[1].plot(hist_t1ce["epoch"], hist_t1ce["val_box_iou"] * 100, marker="s", color="#F97316", lw=2.2, ls="--", label="1-ch T1ce only")
axes[1].set_title("(b) Validation Mean Box IoU theo Epoch", fontweight="bold"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Box IoU (%)"); axes[1].legend(); axes[1].grid(True, alpha=0.4)
plt.tight_layout()
fig.savefig(out_root / "convergence_3ch_vs_t1ce.png", dpi=200)
plt.show()

# 3. Lưới minh họa định tính trên 6 lát cắt có u của Tập Test
test_ds_3 = build_brats_ds("test", False)
test_ds_1 = T1ceOnlyDataset(build_brats_ds("test", False))
tumor_idxs = [i for i, r in enumerate(test_ds_3.records) if int(r["has_tumor"]) == 1 and r.get("gt_box") is not None]
picks = np.linspace(0, len(tumor_idxs) - 1, 6, dtype=int)
model_3ch.eval(); model_t1ce.eval()
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
with torch.no_grad():
    for ax, p_idx in zip(axes.ravel(), picks):
        idx = tumor_idxs[p_idx]
        s3, s1 = test_ds_3[idx], test_ds_1[idx]
        o3 = model_3ch(s3["image_224"].unsqueeze(0).to(CFG["device"]), return_dict=True)
        o1 = model_t1ce(s1["image_224"].unsqueeze(0).to(CFG["device"]), return_dict=True)
        b3 = o3["pred_boxes"][0].cpu().numpy() * 1024.0
        b1 = o1["pred_boxes"][0].cpu().numpy() * 1024.0
        gt = s3["gt_box"].numpy()
        iou3 = box_iou_score(b3, gt)
        iou1 = box_iou_score(b1, gt)
        img = s3["image_1024"].permute(1, 2, 0).numpy()
        ax.imshow(np.clip(img, 0, 1))
        for box, col, ls, lb in ((gt, "#22C55E", "-", "GT"), (b3, "#38BDF8", "-", f"3ch ({iou3:.2f})"), (b1, "#F97316", "--", f"T1ce ({iou1:.2f})")):
            ax.add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], lw=2.2, ec=col, fc="none", ls=ls, label=lb))
        ax.set_title(f"3-ch IoU={iou3:.2f} | T1ce IoU={iou1:.2f}", fontweight="bold", fontsize=10)
        ax.legend(loc="lower right", fontsize=7.5); ax.axis("off")
plt.suptitle("Minh họa Đóng khung Khối u (BraTS2020 Test): GT (Xanh lá) vs 3-ch (Xanh dương) vs T1ce (Cam)", fontweight="bold", fontsize=12)
plt.tight_layout()
fig.savefig(out_root / "qualitative_3ch_vs_t1ce.png", dpi=180)
plt.show()

---
## Cell 9: Tự động Điền `BAO_CAO_TUAN3.md`, Copy sang Drive Nhóm & Đóng gói `.zip` Tải về

In [ ]:
def fmt_delta(a, b, pct=True):
    d = a - b
    return f"{d:+.2f}%" if pct else f"{d:+.4f}"

report_md = f"""# BÁO CÁO TUẦN 3 — Lâm Gia Bảo (Dataset 4: BraTS2020)

## 1. Cấu hình Thực nghiệm
- **Dataset:** BraTS2020 (`42.034` lát cắt — `29.772` Train / `6.087` Val / `6.175` Test; `22.814` u + `19.220` lành; 0% rò rỉ bệnh nhân).
- **Mô hình:** `ViT-LoRA (r=8, α=16)` + `ViTDetBoxHead` (`GIoU + L1 Loss`), `Focal Loss (γ=2.0, ε=0.05)`, bài toán `binary` (2 lớp).
- **Tham số khả huấn:** `{test_3ch['params']['trainable']:,} / {test_3ch['params']['total']:,}` (`{test_3ch['params']['trainable_pct']:.2f}%`).

## 2. Kết quả Đối sánh trên Tập Test (`6.175` lát cắt)

| Chỉ số | 3 kênh `[FLAIR, T1ce, T2]` | 1 kênh `T1ce` đơn lẻ | Chênh lệch (3-ch − 1-ch) |
|---|---:|---:|---:|
| **Best Epoch** | {r3['Best_Epoch']} | {r1['Best_Epoch']} | — |
| **Test Loss** | {r3['Test_Loss']:.5f} | {r1['Test_Loss']:.5f} | {r3['Test_Loss'] - r1['Test_Loss']:+.5f} |
| **Accuracy (%)** | **{r3['Accuracy (%)']:.2f}%** | {r1['Accuracy (%)']:.2f}% | {fmt_delta(r3['Accuracy (%)'], r1['Accuracy (%)'])} |
| **Recall / Sensitivity (%)** | **{r3['Recall (%)']:.2f}%** | {r1['Recall (%)']:.2f}% | {fmt_delta(r3['Recall (%)'], r1['Recall (%)'])} |
| **Specificity (%)** | **{r3['Specificity (%)']:.2f}%** | {r1['Specificity (%)']:.2f}% | {fmt_delta(r3['Specificity (%)'], r1['Specificity (%)'])} |
| **F1-Score (%)** | **{r3['F1 (%)']:.2f}%** | {r1['F1 (%)']:.2f}% | {fmt_delta(r3['F1 (%)'], r1['F1 (%)'])} |
| **ROC-AUC** | **{r3['ROC_AUC']:.4f}** | {r1['ROC_AUC']:.4f} | {fmt_delta(r3['ROC_AUC'], r1['ROC_AUC'], pct=False)} |
| **Mean Box IoU (%)** | **{r3['Mean_Box_IoU (%)']:.2f}%** | {r1['Mean_Box_IoU (%)']:.2f}% | {fmt_delta(r3['Mean_Box_IoU (%)'], r1['Mean_Box_IoU (%)'])} |
| **Box IoU@0.50 (%)** | **{r3['IoU@0.50 (%)']:.2f}%** | {r1['IoU@0.50 (%)']:.2f}% | {fmt_delta(r3['IoU@0.50 (%)'], r1['IoU@0.50 (%)'])} |
| **Box IoU@0.75 (%)** | **{r3['IoU@0.75 (%)']:.2f}%** | {r1['IoU@0.75 (%)']:.2f}% | {fmt_delta(r3['IoU@0.75 (%)'], r1['IoU@0.75 (%)'])} |
"""

(out_root / "BAO_CAO_TUAN3.md").write_text(report_md, encoding="utf-8")
print(report_md)

# Copy sang DRIVE_TEAM & tạo file zip tải về
team_dir = Path(DRIVE_TEAM)
team_dir.mkdir(parents=True, exist_ok=True)
files_to_export = [
    out_root / "BAO_CAO_TUAN3.md",
    out_root / "comparison_3ch_vs_t1ce.csv",
    out_root / "comparison_3ch_vs_t1ce.json",
    out_root / "comparison_3ch_vs_t1ce.png",
    out_root / "convergence_3ch_vs_t1ce.png",
    out_root / "qualitative_3ch_vs_t1ce.png",
    OUT_3CH / "history.csv", OUT_3CH / "test_metrics.json", OUT_3CH / "curves_3ch.png", OUT_3CH / "best_lora.pt",
    OUT_T1CE / "history.csv", OUT_T1CE / "test_metrics.json", OUT_T1CE / "curves_t1ce.png", OUT_T1CE / "best_lora.pt",
]
for p in files_to_export:
    if p.exists():
        dst = team_dir / (f"{p.parent.name}_{p.name}" if p.parent != out_root else p.name)
        shutil.copy2(p, dst)
        print(f"  ✅ Copied {dst.name} ({dst.stat().st_size / 1e6:.2f} MB)")

zip_path = out_root / "week3_brats2020_bao_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_export:
        if p.exists() and p.suffix != ".pt":
            zf.write(p, p.relative_to(out_root))
print(f"\n🎉 Đóng gói thành công: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))